# Topic 03 pilot — How can four-bit weights preserve the output?

**An engineering investigation, not a catalogue of quantizers.** Work on three independently checkable skills:

1. **Memory:** derive scales and implement grouped quantization, explain AWQ/GPTQ mechanisms and output error and distinguish lower-precision values from genuinely packed storage.
2. **Experiment:** compare one pinned model in FP16/NF4, then change only output length from 24 to 96. Keep raw timings, identity and separate memory counters.
3. **Decision:** make a bounded keep/revert/reject-both decision using memory, time and natural answers; explain what the experiment does not establish.

Two short quantization functions, a group-size comparison and one GPU workload edit are yours. Infrastructure is a separate inspectable Python file, not an embedded implementation blob. **The starter intentionally fails tests until you complete those edits.** No leaderboard or grading contract changes.

Read [Book 03](https://levshaazz.github.io/llm-serving-mastery/en/book/03/). Download the [complete lab bundle](https://levshaazz.github.io/llm-serving-mastery/downloads/topic03-pilot.zip). In Colab, upload `topic03_pilot.py` and `topic03-real-layer.npz` through the Files pane. The topic page links the separately recorded technical rehearsal; it does not certify learner understanding.

No GPU is needed for Part A. Part B needs a Linux CUDA GPU that you arrange yourself; availability is not guaranteed. CPU success and a teacher's recording are not your own GPU evidence.


## A1. One matrix, four bits per weight

Central question: given W and typical inputs, how can we preserve its outputs with four bits per weight? Use the local explanation below; the linked Book sections give deeper examples. W[out,in]=[2,4]; each input has four features; each output has two. These invented numbers are not model weights.

The provided scalar operation uses signed codes −8…7 and nearest-even rounding. Derive a symmetric range-fitting step max(abs(weights))/7: seven positive codes must reach the largest magnitude. The fitted range leaves code −8 unused; zero point is zero. Use scale 1 for an all-zero group.

Predict the reconstruction of 0.49 when the same matrix contains 7. Then derive the scale from the range, not from a supplied constant.


### Weight, code and reconstruction

Let s be the positive step in weight units. Divide w by s to obtain a dimensionless number, round to an integer code q, clip to the allowed range, then reconstruct the weight as q × s. Only q needs four bits; its group also needs a scale. Our CPU matrix multiplication uses reconstructed floating-point weights, not packed codes.

For global maximum 7, s = 7/7 = 1. Worked example: w = −0.6 gives normalized value −0.6, code −1, restored weight −1 and signed error −1 − (−0.6) = −0.4. Now do **0.49 yourself**, before running tests. At an exact half-integer, nearest-even rounding gives 0.5 → 0 and 1.5 → 2.

References: [representation and scale](https://levshaazz.github.io/llm-serving-mastery/en/book/03/#representation), then [matrix and groups](https://levshaazz.github.io/llm-serving-mastery/en/book/03/#matrix-example).


In [ ]:
import math
WEIGHTS = [[0.49,0.4,0.6,7],[-0.31,-0.6,0.92,-1.4]]
INPUTS = [[1., 1., 0., 0.], [0., 1., 0., 0.]]
HELD_OUT = [[10., 0., 0., 1.]]
PREDICTION = {
    'weight_0_49_with_global_scale': 'TODO: predict and explain',
    'group_size_2_error_and_metadata': 'TODO: predict the trade-off',
    'will_4bit_necessarily_run_faster': 'TODO: reason',
    'timing_at_24_vs_96_ids': 'TODO: predict the ranking/gap',
}
def quantize_scalar(value, scale, zero_point=0):
    """Provided signed INT4; nearest-even, clip [-8,7], reconstruct.
    Absmax/7 fits a symmetric range, leaving stored code -8 unused. Not NF4.
    """
    if not math.isfinite(value) or not math.isfinite(scale) or scale <= 0:
        raise ValueError('finite value and positive finite scale required')
    if type(zero_point) is not int or not -8 <= zero_point <= 7:
        raise ValueError('signed INT4 zero point required')
    code = max(-8, min(7, round(value / scale) + zero_point))
    return code, scale * (code - zero_point)
def outputs(weights, inputs):
    """[samples,in] times transpose([out,in]) -> [samples,out], no bias."""
    return [[sum(w*x for w, x in zip(row, vector)) for row in weights] for vector in inputs]
def compare(weights, quantized, inputs):
    """Provided MSE/accounting helper. Mean over ALL scalar entries.
    Ideal storage assumes FP16 scales; arithmetic uses exact Python scales.
    Scale rounding, alignment, headers and Python-object sizes are omitted.
    """
    restored = quantized['restored']
    error = [[a-b for a, b in zip(row, original)] for row, original in zip(restored, weights)]
    signed = outputs(error, inputs)
    count = sum(map(len, weights))
    return dict(weight_mse=sum(v*v for row in error for v in row)/count,
                signed_output_error=signed,
                output_mse=sum(v*v for row in signed for v in row)/(len(inputs)*len(weights)),
                ideal_code_bytes=(count+1)//2, fp16_scale_bytes=2*len(quantized['scales']))


In [ ]:
def derive_scale(values):
    """Return max(abs(values))/7, or 1.0 for an all-zero group.
    STUDENT: derive the step from seven positive codes and the group's range.
    """
    if not values or any(not math.isfinite(v) for v in values):
        raise ValueError('nonempty finite values required')
    # TODO 1: derive the scale, including the all-zero case.
    raise NotImplementedError('Derive a scale from the weights')


In [ ]:
def quantize_groups(weights, group_size=None):
    """Return {'restored': rectangular matrix, 'scales': flat list}.
    None: ONE scale for the complete matrix. Positive integer: contiguous chunks
    within EACH output row; never cross rows. Keep the shorter final group.
    For our 2x4 matrix: size 4 = per-output-channel; size 2 = two groups per row.
    Use derive_scale and quantize_scalar. Do not mutate weights. This returns
    reconstructed floats, not physically packed codes.
    """
    if not weights or not weights[0] or any(len(row) != len(weights[0]) for row in weights):
        raise ValueError('nonempty rectangular [out,in] matrix required')
    if any(not math.isfinite(v) for row in weights for v in row):
        raise ValueError('finite weights required')
    if group_size is not None and (type(group_size) is not int or group_size < 1):
        raise ValueError('positive integer group size or None required')
    # TODO 2: partition, derive each scale, restore every weight.
    raise NotImplementedError('Compare a global scale with row-local groups')


In [ ]:
def test_scale(function):
    for values, expected in [([.49,.4,.6,7],1.), ([-1.4,.9],.2),
                             ([0.,0.],1.), ([-7.],1.), ([.49,.4],.07)]:
        assert math.isclose(function(values), expected, abs_tol=1e-12), values
    for values in [[], [float('nan')], [float('inf')]]:
        try:
            function(values)
        except ValueError:
            continue
        raise AssertionError('Invalid range must fail')
def test_groups(function):
    import copy
    original = copy.deepcopy(WEIGHTS)
    whole = function(WEIGHTS, None)
    assert whole['restored'] == [[0.,0.,1.,7.], [0.,-1.,1.,-1.]]
    expected_scales = {4:[1.,.2], 2:[.07,1.,.6/7,.2], 1:[abs(w)/7 for row in WEIGHTS for w in row]}
    for size, expected in expected_scales.items():
        result = function(WEIGHTS, size)
        assert len(result['scales']) == len(expected)
        assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(result['scales'],expected))
    # Check EVERY reconstructed weight, independently of the returned scales.
    # Different row ranges expose global restoration disguised as row-local.
    for matrix in [WEIGHTS, [[.1,.2,7.],[-.4,-.5,-.6]], [[0.,0.,0.],[1.,-2.,3.]]]:
        before = copy.deepcopy(matrix)
        for size in [None, 1, 2, len(matrix[0]), len(matrix[0])+1]:
            result = function(matrix, size)
            assert len(result['restored']) == len(matrix)
            expected_count = 1 if size is None else len(matrix)*math.ceil(len(matrix[0])/size)
            assert len(result['scales']) == expected_count, 'Each row owns its tail scale'
            global_max = max(abs(v) for row in matrix for v in row)
            scale_index = 0
            for row, restored in zip(matrix, result['restored']):
                assert len(restored) == len(row)
                width = size or len(row)
                for start in range(0, len(row), width):
                    values = row[start:start+width]
                    maximum = global_max if size is None else max(abs(v) for v in values)
                    scale = maximum/7 if maximum else 1.
                    actual_scale = result['scales'][0 if size is None else scale_index]
                    assert math.isclose(actual_scale, scale, rel_tol=1e-12, abs_tol=1e-12)
                    for offset, value in enumerate(values):
                        expected = max(-8, min(7, round(value/scale)))*scale
                        assert math.isclose(restored[start+offset], expected, rel_tol=1e-12, abs_tol=1e-12), 'Wrong group reconstruction'
                    scale_index += 1
            assert matrix == before, 'Do not mutate any input matrix'
    group = function(WEIGHTS, 2)
    assert math.isclose(group['restored'][0][1], .42, abs_tol=1e-12)
    # Output reduction is checked from all restored entries, not a magic loss.
    probe = [[1.,2.,3.,4.], [-1.,0.,2.,-3.]]
    for result in [whole, group]:
        expected = [[sum((q-w)*x for q,w,x in zip(qrow,wrow,vector))
                     for qrow,wrow in zip(result['restored'], WEIGHTS)] for vector in probe]
        measured = compare(WEIGHTS, result, probe)
        assert measured['signed_output_error'] == expected
        assert math.isclose(measured['output_mse'], sum(v*v for row in expected for v in row)/4)
    tail = function([[.1,.2,7.],[.4,.5,.6]], 2)
    assert len(tail['scales']) == 4, 'Tail group cannot cross rows'
    assert [len(row) for row in tail['restored']] == [3,3]
    assert function([[0.,0.]], 2) == {'restored':[[0.,0.]], 'scales':[1.]}
    assert WEIGHTS == original, 'Do not mutate the source matrix'
    for weights,size in [([],2),([[]],2),([[1],[1,2]],2),([[1]],0),([[1]],True),([[float('nan')]],2)]:
        try:
            function(weights,size)
        except ValueError:
            continue
        raise AssertionError('Invalid matrix/group must fail')

test_scale(derive_scale)
test_groups(quantize_groups)
print('Range and group tests passed; now explain the output errors.')


## A2. Compare the SAME matrix, then change only the inputs

Required: fill COMPARISON_SIZES with per-tensor (None), per-output-row (4), groups of 2, and groups of 1. Predict before adding each size. Each group stays inside one output row, including a shorter final group. No calibration inputs are used to choose these baseline scales.

For EACH variant, report scales, reconstructed W, weight MSE, both signed output-error vectors, output MSE, packed-code bytes and scale bytes. The supplied helper divides output squared error by samples × outputs = 4. Then repeat on HELD_OUT without changing the scales.

Arithmetic uses full Python scales. Storage accounting assumes two bytes per FP16 scale; it omits FP16 scale-rounding error, headers and alignment. Group size 1 is almost exact in this arithmetic but costs 4+16=20 bytes, more than the original 16! A float matrix of reconstructed values is not packed storage.

Explain: which errors survive with groups of 2? Why does a different input change output error while weight MSE stays fixed?


In [ ]:
COMPARISON_SIZES = [None]  # TODO: add 4, 2, 1; keep the per-tensor baseline.
assert COMPARISON_SIZES == [None, 4, 2, 1], 'Compare the four required granularities'
CPU_COMPARISON = {}
for size in COMPARISON_SIZES:
    quantized = quantize_groups(WEIGHTS, size)
    CPU_COMPARISON[str(size)] = dict(
        scales=quantized['scales'], restored=quantized['restored'],
        calibration=compare(WEIGHTS, quantized, INPUTS),
        held_out=compare(WEIGHTS, quantized, HELD_OUT))
print(CPU_COMPARISON)


In [ ]:
TOY_EXPLANATION = {
    'group2_remaining_error': 'TODO: identify a weight and its surviving error',
    'changed_input_same_weights': 'TODO: explain which input amplifies which error',
    'group1_storage_tradeoff': 'TODO: compare code bytes plus scales with original FP16',
}


## A3. Explain AWQ and GPTQ BEFORE the model run

Follow the numerical mechanisms below, then fill METHOD_EXPLANATION in your own words. Understanding how inputs affect weight choices is required; implementing full AWQ/GPTQ algorithms is optional. Keep format, calibration algorithm and execution kernel separate.


### AWQ: preserving a function before changing its approximation

An input channel means a **feature column**, not an output neuron or a vocabulary token. Here use the explicitly different inputs [[10,1,0,0],[0,1,0,0]]. They make feature 0 more active than feature 1. Multiply column 0 of W by a and divide column 0 of X by a. In every dot product the factors cancel before rounding. For the first output and first input, 10 × 0.49 + 1 × 0.4 = 5.3; with a = 2, 5 × 0.98 + 1 × 0.4 = 5.3.

Round the scaled weights and use the inverse-scaled inputs. With global step 1, column-0 weight 0.49 used to become 0; after scaling it becomes 0.98 → 1, equivalent to 0.5 in the original coordinates. Its contribution is much closer, but other weights still round. Too large an a can become the new maximum, enlarge the shared step and damage other channels. The toy is an **AWQ-style mechanism**, not the full AWQ scale-search algorithm. Activations guide the choice; large weight magnitude alone does not establish importance.

Use the [AWQ figure and factors 1/2/4/32](https://levshaazz.github.io/llm-serving-mastery/en/book/03/#awq) to explain why a = 32 loses to a = 4.

### GPTQ: compensating with another weight

Keep step 1 fixed; take w = [0.49,0.40] and X = [[1,1],[0,1]]. The two columns co-occur in one example; the second also appears alone. Their **uncentered Gram** XᵀX is [[1,1],[1,2]], not a Pearson correlation matrix. Rounding the first weight to 0 removes 0.49 from the first output. Adding d to the second weight gives signed errors [−0.49+d,d]. Their squared sum (−0.49+d)²+d² is minimized at d = 0.49/2 = 0.245, so 0.40+d = 0.645.

Then 0.645 rounds to 1. Compare original outputs [0.89,0.40], independently rounded outputs [0,0], and compensated outputs [1,1]. Output SSE falls from 0.9521 to 0.3721 while weight SSE rises from 0.4001 to 0.6001. The second example gets worse although total output SSE improves. Explain this with the [GPTQ lattice and error ellipses](https://levshaazz.github.io/llm-serving-mastery/en/book/03/#gptq). Full GPTQ extends compensation to remaining coordinates, with damping and efficient updates; this example teaches its objective rather than implementing the algorithm.

### NF4: a nonuniform alphabet plus a block scale

NF4 stores one of **16 reconstruction levels** in a four-bit index. Levels are closer together near zero than at the ends. They are reconstruction values, not quantile boundaries. Divide each block by its absolute maximum to enter [−1,1], choose the nearest codebook value, then multiply by that same maximum to reconstruct. Thus the same index can reconstruct different absolute weights in differently scaled blocks. All-zero blocks reconstruct zero without division by zero.

Read the [16-level figure and block normalization](https://levshaazz.github.io/llm-serving-mastery/en/book/03/#packing). NF4's design uses a normal-distribution motivation; our measured Qwen weights need not be normally distributed. Check distribution fit rather than assuming it. The CUDA run uses NF4, **not AWQ or GPTQ**.


In [ ]:
METHOD_EXPLANATION = {
    'awq_before_rounding': 'TODO: explain why 5.3 stays 5.3 and what changes after rounding',
    'awq_large_factor': 'TODO: explain the new maximum and the error at factor 32',
    'gptq_0_645': 'TODO: derive the correction from the two input columns',
    'gptq_individual_input': 'TODO: compute both signed errors and identify the worsened input',
    'nf4_block_scale': 'TODO: distinguish index, normalized level and restored weight',
    'format_algorithm_kernel': 'TODO: distinguish the three and name what the GPU run uses',
}


## A4. A real Qwen matrix and user-content activations

Download/upload `topic03-real-layer.npz` from the matching bundle. No model download or GPU is needed here. This is **not invented Gaussian data**: fixed rows 0–63 of Qwen block 12's MLP down projection, W[out,in]=[64,4864]. The original matrix has 896 output rows. X contains 16 evenly spaced user-content positions per prompt, chosen before inspecting activations. Six calibration prompts and six held-out prompts cover arithmetic, code, JSON extraction, Russian, support and summarization. Token IDs and exact texts are in the [capture](https://levshaazz.github.io/llm-serving-mastery/data/topic03-calibration-study.json).

Full-layer figures in Book use all 896 rows and all user positions. Your bounded slice will have DIFFERENT errors. Each position has 4864 intermediate features, not 4864 tokens. The shared chat wrapper is excluded from this lab's X; it remains part of the context that produced the activations.

Use your two tested functions on these real weights. Keep the baseline, add row/128/64/32. Implement output error as the Frobenius norm of the difference divided by the reference norm. Select the smallest calibration error subject to **at most 4.30 ideal bits per weight**, including FP16 scales. This budget is an exercise constraint, not a production threshold.

Then investigate a different choice: global clipping at the predeclared percentiles 100/99/99.5. The supplied helper reduces the range before rounding: smaller steps preserve central weights better but saturate tails. Compare the weight-MSE winner with the calibration-output winner; select using outputs. This is a separate branch, not a jointly optimized group/clipping recipe. The supplied block64 NF4 helper also acts on this exact W and X; it tests a nonuniform codebook, not a GPU kernel. Freeze BOTH student choices before loading held-out arrays.


In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import matplotlib.pyplot as plt

REAL_PATH = next((p for p in [Path('topic03-real-layer.npz'), Path('public/downloads/topic03-real-layer.npz'), Path('../public/downloads/topic03-real-layer.npz')] if p.is_file()), None)
assert REAL_PATH is not None, 'Upload topic03-real-layer.npz from the matching bundle'
assert hashlib.sha256(REAL_PATH.read_bytes()).hexdigest() == '08b6ede8cb3f790301fb76b251e2e8ccf05977cefeb8f1a754072537f9d688fa'
REAL = np.load(REAL_PATH, allow_pickle=False)
W_REAL = REAL['weights'].astype(np.float64)
X_CAL = np.concatenate([REAL['x_'+str(i)] for i in range(6)]).astype(np.float64)
assert W_REAL.shape == (64, 4864) and X_CAL.shape == (96, 4864)
Y_CAL = X_CAL @ W_REAL.T
assert Y_CAL.shape == (96, 64)
print('W [output features,input features]:', W_REAL.shape)
print('X [user positions,input features]:', X_CAL.shape, 'Y [positions,output features]:', Y_CAL.shape)


### A4a. Measure what the layer does

Each of the 96 rows is one sampled user position; each of the 64 columns of Y is one output feature. Y = X × Wᵀ contracts the 4864 input features. The **Frobenius norm** treats all 96 × 64 entries as one vector: square each entry, sum the squares, take the square root.

Relative output error is norm(prediction − reference) / norm(reference). The denominator makes it dimensionless and compares the difference with reference output energy. If the error vector is [3,4] and the reference is [0,10], their norms are 5 and 10, giving relative error 0.5. Neither the number of positions nor the number of features belongs in this ratio as an extra divisor. For a zero reference, the relative metric is undefined; reject it.

A result of 0.12 means a difference norm equal to 12% of the reference norm. It is **not 12% wrong answers**, a token error rate or a mean of per-prompt percentages. High-energy outputs contribute more to the pooled metric. Complete one short expression and test it before comparing groups. [Calibration and output-error reference](https://levshaazz.github.io/llm-serving-mastery/en/book/03/#methods).


In [ ]:
def relative_output_error(prediction, reference):
    assert prediction.shape == reference.shape
    assert np.all(np.isfinite(prediction)) and np.all(np.isfinite(reference))
    assert np.linalg.norm(reference) > 0, 'Relative error needs nonzero reference energy'
    output_error = None  # TODO: relative Frobenius error, not weight MSE
    assert output_error is not None, 'Measure the layer output, not only weights'
    return float(output_error)

assert np.isclose(relative_output_error(np.array([[3.,14.]]), np.array([[0.,10.]])), .5)
assert np.isclose(relative_output_error(Y_CAL, Y_CAL), 0)
try:
    relative_output_error(np.zeros((1,2)), np.zeros((1,2)))
except AssertionError:
    pass
else:
    raise AssertionError('Zero denominator must be rejected')

REAL_SIZES = [None]  # TODO: add 4864, 128, 64, 32, in that order
assert REAL_SIZES == [None, 4864, 128, 64, 32]
REAL_RESULTS, REAL_RESTORED = [], {}
for size in REAL_SIZES:
    packed = quantize_groups(W_REAL.tolist(), size)
    restored = np.asarray(packed['restored'])
    prediction = X_CAL @ restored.T
    output_error = relative_output_error(prediction, Y_CAL)
    assert np.isclose(output_error, np.linalg.norm(prediction-Y_CAL)/np.linalg.norm(Y_CAL))
    REAL_RESTORED[str(size)] = restored
    REAL_RESULTS.append(dict(group=size, weight_mse=float(np.mean((restored-W_REAL)**2)),
        calibration_error=float(output_error), ideal_bits_per_weight=4+16*len(packed['scales'])/W_REAL.size))
print(json.dumps(REAL_RESULTS, indent=2))


### A4b. Choose under the storage budget

Read your table: remove settings above 4.30 ideal bits/weight, then choose the lowest calibration output error among the remaining settings. Four code bits plus sixteen scale bits per group gives 4.25 at group 64 and 4.5 at group 32. Lower error alone cannot pick 32. These are ideal FP16-scale counts, not measured GPU memory. Do not inspect held-out to break a tie or tune the budget.


In [ ]:
CHOSEN_GROUP = None  # TODO: choose using calibration and the declared budget only
eligible = [r for r in REAL_RESULTS if r['ideal_bits_per_weight'] <= 4.30]
chosen_row = next((r for r in eligible if r['group'] == CHOSEN_GROUP), None)
assert chosen_row is not None, 'Your choice must be eligible'
assert all(chosen_row['calibration_error'] <= r['calibration_error'] for r in eligible)
print('Chosen group:', CHOSEN_GROUP, chosen_row)


### A4c. A separate clipping investigation

The supplied helper clips large weights to a percentile-derived bound before rounding. A smaller bound reduces the step but adds saturation error in the tails. Compare the three **predeclared** global settings 100/99/99.5. Choose by calibration outputs and explain why weight MSE can prefer a different setting. This branch uses one global scale; it does not combine your chosen groups with clipping.


In [ ]:
# A separate global clipping experiment: predefined candidates, not jointly tuned groups.
# SUPPLIED numerical helper. Symmetric 15-level grid (-7..7); float64 QDQ.
CLIP_CANDIDATES = (100, 99, 99.5)
def clipped_qdq(weights, percentile):
    bound = float(np.percentile(np.abs(weights), percentile))
    scale = bound/7 if bound else 1.0
    return np.clip(np.rint(weights/scale), -7, 7)*scale

CLIP_RESTORED = {p: clipped_qdq(W_REAL, p) for p in CLIP_CANDIDATES}
CLIP_RESULTS = [dict(percentile=p, weight_mse=float(np.mean((q-W_REAL)**2)),
    calibration_error=float(np.linalg.norm(X_CAL @ q.T-Y_CAL)/np.linalg.norm(Y_CAL)))
    for p,q in CLIP_RESTORED.items()]
WEIGHT_CHOICE = min(CLIP_RESULTS, key=lambda r:r['weight_mse'])['percentile']
CHOSEN_CLIP = None  # TODO: select by calibration output error, then explain the conflict
assert CHOSEN_CLIP == min(CLIP_RESULTS, key=lambda r:r['calibration_error'])['percentile']
print('Clipping: weight choice', WEIGHT_CHOICE, 'output choice', CHOSEN_CLIP, CLIP_RESULTS)


### A4d. NF4 on the same weights and inputs

The supplied helper reshapes each output row into blocks of 64 input weights, computes each block's absmax, normalizes, selects the nearest of the 16 levels, and restores. Read those operations in the code. Compare with **uniform group 64** on the same W and X. This is a codebook comparison, not proof of a fast CUDA implementation. NF4's ideal FP32 absmax costs 4.5 bits/weight at block 64, outside the separate 4.30-bit uniform exercise budget; these formats have different metadata costs.


In [ ]:
# SUPPLIED NF4 nearest-codebook simulation on EXACTLY the same real weights.
# Row-local block 64 divides 4864. Float64 absmax, no packing or GPU kernel.
NF4_LEVELS = np.asarray([-1,-0.6961928009986877,-0.5250730514526367,-0.39491748809814453,-0.28444138169288635,-0.18477343022823334,-0.09105003625154495,0,0.07958029955625534,0.16093020141124725,0.24611230194568634,0.33791524171829224,0.44070982933044434,0.5626170039176941,0.7229568362236023,1], dtype=np.float64)
def nf4_qdq(weights, block=64):
    assert weights.ndim == 2 and weights.shape[1] % block == 0
    blocks = weights.reshape(weights.shape[0], -1, block)
    absmax = np.max(np.abs(blocks), axis=-1, keepdims=True)
    safe = np.where(absmax == 0, 1.0, absmax)
    normalized = blocks/safe
    codes = np.argmin(np.abs(normalized[...,None]-NF4_LEVELS), axis=-1)
    return (NF4_LEVELS[codes]*absmax).reshape(weights.shape)
NF4_RESTORED = nf4_qdq(W_REAL)
NF4_CAL_ERROR = float(np.linalg.norm(X_CAL @ NF4_RESTORED.T-Y_CAL)/np.linalg.norm(Y_CAL))
print('Same-matrix uniform g64 / NF4 calibration errors:',
      next(r['calibration_error'] for r in REAL_RESULTS if r['group']==64), NF4_CAL_ERROR)
print('Ideal format ledger: uniform FP16 scale 4.25; NF4 FP32 absmax 4.5 bits/weight at block64.')
print('CPU simulations keep unrounded float64 scales; these are not actual GPU memory counts.')


### A4e. Freeze both choices

Record the group and separately selected clipping percentile **before** opening held-out activations. Freeze the candidate list and budget too. If you later revise the experiment, keep the old results and label the new investigation; do not rewrite a test-guided choice as calibration.


In [ ]:
FROZEN_CHOICE = json.dumps(dict(group=CHOSEN_GROUP, clipping=CHOSEN_CLIP,
    clipping_candidates=CLIP_CANDIDATES, budget=4.30,
    criterion='calibration relative output error; separate group and clipping experiments'), sort_keys=True)
print('Choice frozen BEFORE held-out:', FROZEN_CHOICE)


## A5. Show the distributions and test transfer

Build histograms of the original and reconstructed weights on **the same bin edges and axes**. Show the full range first, then a labelled central zoom if useful. A smooth-looking histogram is not evidence of accurate outputs. On the scatterplot, each point is one position/output coordinate; it is not one vocabulary token.

The separate clipping branch tests only 100/99/99.5 on the same matrix. Compare the weight-MSE winner with the calibration-output winner. Explain why they differ, then freeze both branch choices. This is not a joint group-plus-clipping optimization. The supplied NF4 helper uses the same W and X, block 64 and an explicit codebook: compare its errors with uniform group64 before the whole-model GPU run. It does not reproduce FP32 CUDA normalization or packed storage.

Only now load held-out. Report group, clipping and NF4 errors without reselecting. A disappointing result is a finding, not permission to tune on the test. The split is public for teaching, not a secret judge. Compare errors per prompt as well as pooled errors: the pooled norm weights high-energy outputs more strongly; it is not a mean of prompt percentages.


In [ ]:
PLOT_GROUPS = [None]  # TODO: add your frozen choice for a matched comparison
assert PLOT_GROUPS == [None, CHOSEN_GROUP]
edges = np.linspace(-np.max(np.abs(W_REAL)), np.max(np.abs(W_REAL)), 61)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
for ax, values, title in zip(axes, [W_REAL, *[REAL_RESTORED[str(g)] for g in PLOT_GROUPS]], ['Original', 'Tensor', 'Chosen group']):
    ax.hist(values.ravel(), bins=edges, weights=np.ones(values.size)/values.size*100)
    ax.set(xlabel='Weight value', ylabel='Weights per bin (%)', title=title)
fig.tight_layout(); fig.savefig('topic03-weights-full.png', dpi=150); plt.show()

# Provided matched center zoom. Keep the full-range plots above and disclose tails.
zoom_edges = np.linspace(-.05, .05, 61)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
for ax, values, title in zip(axes, [W_REAL, *[REAL_RESTORED[str(g)] for g in PLOT_GROUPS]], ['Original', 'Tensor', 'Chosen group']):
    outside = np.mean((values < zoom_edges[0]) | (values > zoom_edges[-1]))*100
    ax.hist(values.ravel(), bins=zoom_edges, weights=np.ones(values.size)/values.size*100)
    ax.set(xlabel='Weight value; center zoom [-.05,.05]', ylabel='% of ALL weights per bin',
           title=title+'; outside window '+str(round(outside,2))+'%')
fig.tight_layout(); fig.savefig('topic03-weights-center.png', dpi=150); plt.show()
print('Zero fractions:', {str(g): float(np.mean(REAL_RESTORED[str(g)] == 0)) for g in PLOT_GROUPS})


### A5a. Test the frozen setting

Load the six held-out prompts and keep both choices unchanged. The scatter uses 500 evenly spaced position/output coordinates: a point on the diagonal has the same original and reconstructed value. Distance from the diagonal reveals a discrepancy, not a vocabulary-token mistake. Save the distributions and scatter, then answer the reading questions below.


In [ ]:
assert json.loads(FROZEN_CHOICE)['group'] == CHOSEN_GROUP, 'Do not change the frozen choice'
assert json.loads(FROZEN_CHOICE)['clipping'] == CHOSEN_CLIP
assert tuple(json.loads(FROZEN_CHOICE)['clipping_candidates']) == CLIP_CANDIDATES
X_TEST = np.concatenate([REAL['x_'+str(i)] for i in range(6,12)]).astype(np.float64)
Y_TEST = X_TEST @ W_REAL.T
Q_TEST = X_TEST @ REAL_RESTORED[str(CHOSEN_GROUP)].T
HELD_OUT_ERROR = float(np.linalg.norm(Q_TEST-Y_TEST)/np.linalg.norm(Y_TEST))
print('Frozen choice held-out relative output error:', HELD_OUT_ERROR)
CLIP_TEST_ERROR = float(np.linalg.norm(X_TEST @ CLIP_RESTORED[CHOSEN_CLIP].T-Y_TEST)/np.linalg.norm(Y_TEST))
NF4_TEST_ERROR = float(np.linalg.norm(X_TEST @ NF4_RESTORED.T-Y_TEST)/np.linalg.norm(Y_TEST))
print('Separate frozen clipping / NF4 held-out errors:', CLIP_TEST_ERROR, NF4_TEST_ERROR)
for i in range(6,12):
    x = REAL['x_'+str(i)].astype(np.float64)
    y = x @ W_REAL.T
    print('held-out prompt', i, float(np.linalg.norm(x @ REAL_RESTORED[str(CHOSEN_GROUP)].T-y)/np.linalg.norm(y)))
idx = np.linspace(0, Y_TEST.size-1, 500).astype(int)
limit = max(np.max(np.abs(Y_TEST.ravel()[idx])), np.max(np.abs(Q_TEST.ravel()[idx])))
plt.figure(figsize=(5,5)); plt.scatter(Y_TEST.ravel()[idx], Q_TEST.ravel()[idx], s=6)
plt.plot([-limit,limit],[-limit,limit], '--', color='black')
plt.xlim(-limit,limit); plt.ylim(-limit,limit); plt.gca().set_aspect('equal')
plt.xlabel('Original layer output'); plt.ylabel('Reconstructed layer output')
plt.tight_layout(); plt.savefig('topic03-output-scatter.png', dpi=150); plt.show()
LAYER_DECISION = dict(schema='topic03-student-layer-v2', choice=json.loads(FROZEN_CHOICE),
    payload_sha256=hashlib.sha256(REAL_PATH.read_bytes()).hexdigest(), calibration=REAL_RESULTS,
    held_out_error=HELD_OUT_ERROR, clipping=CLIP_RESULTS, clipping_held_out_error=CLIP_TEST_ERROR,
    nf4=dict(blocksize=64, arithmetic_dtype='float64', calibration_error=NF4_CAL_ERROR,
             held_out_error=NF4_TEST_ERROR),
    scope='64 output rows; 16 user positions per prompt; dense QDQ, not GPU performance')
Path('topic03-layer-decision.json').write_text(json.dumps(LAYER_DECISION, indent=2))


In [ ]:
LAYER_EXPLANATION = {
    'metric_0_12': 'TODO: interpret numerator, denominator and axes; exclude accuracy',
    'budget_choice': 'TODO: name the eligible winner and why group 32 is excluded',
    'clipping_conflict': 'TODO: report both winners and explain the differing objectives',
    'histogram_reading': 'TODO: compare zero fractions and tails on matched axes',
    'histogram_vs_output': 'TODO: explain why similar distributions can give different outputs',
    'scatter_and_transfer': 'TODO: read one off-diagonal point and compare calibration/held-out',
}


## A6. Timing statistics are PROVIDED

Run the tests, then reuse the helper. You do not implement a median. Input is seconds; output is milliseconds. Five samples give a median and observed range, not a reliable tail percentile.


In [ ]:
def summarize_seconds(values):
    """PROVIDED: validated median/range in ms; never mutate raw seconds."""
    if not values or any(type(v) not in (int,float) or not math.isfinite(v) or v <= 0 for v in values):
        raise ValueError('nonempty positive finite timings required')
    ordered = sorted(values)
    n = len(ordered)
    median = (ordered[(n-1)//2] + ordered[n//2]) / 2
    return dict(median_ms=1000*median, minimum_ms=1000*ordered[0], maximum_ms=1000*ordered[-1])
def test_summary(function):
    samples = [.004,.001,.002,.003]
    assert function(samples) == {'median_ms':2.5, 'minimum_ms':1., 'maximum_ms':4.}
    assert samples == [.004,.001,.002,.003], 'Do not mutate raw evidence'
    assert function([.001,.003,.002])['median_ms'] == 2.
    for invalid in [[],[0.],[-1.],[float('nan')],[float('inf')],[True]]:
        try:
            function(invalid)
        except ValueError:
            continue
        raise AssertionError('Reject invalid timings')
test_summary(summarize_seconds)


## B1. Write the experimental contract

We compare `Qwen/Qwen2.5-0.5B-Instruct` at immutable revision `7ae557604adf67be50417f59c2c2f167def9a775`. Both use batch 1, greedy generation, SDPA, KV caching, the same tokenizer/chat wrapper and FP16 computation. NF4 uses bitsandbytes, FP16 compute and double quantization off. This changes **representation and execution path**, not bit width alone.

There are two deliberately separate workloads:

- **Timing intervention:** literal prompt `Continue the sequence with comma-separated integers and no explanation: 2, 4, 6,`; force exactly 24 or 96 output IDs. Five raw trials per length and format, after one excluded warmup for each condition. Length order alternates within a format. This is a synthetic timing workload, not a quality answer or support-service latency benchmark.
- **Quality smoke:** twelve handwritten, labelled support tickets (four each: billing/access/bug); natural stopping, maximum 12 IDs. Score an answer only if stripping surrounding whitespace yields the exact lowercase label. Inspect every mismatch. This tiny synthetic set does not certify real-world accuracy. Six public transfer cases are kept out of the default run; they are not a secret evaluation set.

The GPU clock brackets `generate` with CUDA synchronization. It excludes model loading, tokenization and input transfer, and includes host dispatch. No HTTP TTFT, p95 or throughput claim. Formats load sequentially: FP16 then NF4, a time-order confound. A reverse-order repeat is optional only after diagnosing the first run.

**Your intervention:** keep baseline length 24 and change only the second value to 96. Do not change the model, prompt, batch, sampler or package pins at the same time.


### Layer preservation and whole-model execution

**Your reconstructed A4 matrix is not installed into the GPU model.** Part A tests how an approximation preserves one layer's outputs on recorded inputs. Part B loads an entire FP16 model and a separately quantized bitsandbytes NF4 candidate. They share the model family and mechanism question but have different scope, normalization arithmetic and execution. Layer error alone cannot predict ticket accuracy or GPU latency.

Before running either GPU format, declare a **classroom screening rule** for this toy router. Specify a context, a minimum correct-label count out of twelve, a maximum invalid-string count and a rationale. A router sending tickets directly to teams may tolerate different errors than a tool whose proposals a human checks. Choose your rule before seeing results; no universal production threshold or course-judge change is implied. Even twelve successes support only this smoke check.

The exact-label scorer defines how an answer is counted. Your rule defines whether the counted outcome is acceptable for this investigation.


In [ ]:
QUALITY_RULE = {
    'context': 'TODO: who uses the router and what a wrong route costs',
    'min_correct_of_12': None,  # TODO: integer in 0..12, declared before running
    'max_invalid_of_12': None,  # TODO: integer in 0..12, declared before running
    'rationale': 'TODO: justify the classroom rule and its limits',
}
FROZEN_QUALITY_RULE = None  # The readiness check freezes this before any GPU run.


In [ ]:
def check_lengths(lengths):
    assert type(lengths) in (list,tuple) and list(lengths) == [24,96], (
        'Keep baseline 24; change only the second output cap to 96.')

LENGTHS = [24, 24]  # TODO 3: change ONLY the second output length to 96.
check_lengths(LENGTHS)
TRIALS = 5
assert TRIALS == 5


## B2. Inspect and verify the infrastructure file

This file contains the public synthetic cases, safety gates, model setup, clocks, profiler and artifact validator. You do not need to understand all infrastructure at once. Inspect three small entry points and answer a concrete question:

1. `score_output`: is `Billing.` a correct answer? Which transformation is allowed?
2. The nested `generate` inside `run`: where are the two CUDA synchronizations, and which work lies outside the clock?
3. `validate_artifact`: which equality prevents comparing different tokenized prompts across formats?

The SHA below binds this notebook to reviewed bytes. Use the matching bundle; do not bypass a mismatch or fetch an unpinned `main` file.

The runner is standard-library-only on import. Inference executes in a separate process, not in the Colab kernel. Runtime package pins live in an isolated venv using the host Torch/CUDA installation; the base Colab environment is not modified. Colab may lack `ensurepip`, so setup uses `--without-pip` and the inherited installer, after verifying that its install target is the dedicated venv.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, subprocess, sys

RUNNER_CANDIDATES = [Path('topic03_pilot.py'), Path('../teaching/topic03_pilot.py'), Path('teaching/topic03_pilot.py')]
RUNNER = next((path.resolve() for path in RUNNER_CANDIDATES if path.is_file()), None)
assert RUNNER is not None, 'Upload the provided topic03_pilot.py through Colab Files, then rerun this cell.'
EXPECTED_RUNNER_SHA256 = '85e9a4c30ea0e8dde746ed77b8f7fa7cbb38728d094c9c8fc743dc7b7f31ec2e'
assert hashlib.sha256(RUNNER.read_bytes()).hexdigest() == EXPECTED_RUNNER_SHA256, 'Wrong runner revision: use matching bundle.'
spec = importlib.util.spec_from_file_location('topic03_pilot_infrastructure', RUNNER)
infra = importlib.util.module_from_spec(spec)
spec.loader.exec_module(infra)
assert infra.PROTOCOL == 'topic03-engineering-pilot-v1'
print('Verified protocol:', infra.PROTOCOL)
for case in infra.CASES:
    print(case['id'], repr(case['text']), '->', case['label'])
print('Public transfer cases excluded from this run:', len(infra.TRANSFER_CASES))


## B3. Prepare a bounded CUDA run — explicit opt-in

Startup requires an otherwise idle GPU, at least **5 GiB free host RAM**, **5 GiB free GPU memory**, and 3 GiB free cache-disk space if downloading is needed. During work: at least 2 GiB free host RAM and 1 GiB free GPU memory. These are conservative gates for this new 0.5B pilot, **not** relaxed gates for the historical v3 protocol. They are sampled headroom checks, not an OS memory sandbox or a guarantee on every Colab machine.

Reuse the pinned model cache. Set `LSM_MODEL_CACHE` only if you know your existing cache path; otherwise the normal Hugging Face cache is reused. `HF_HUB_OFFLINE=1` refuses missing weights. Do not delete models between attempts, mount personal Drive unnecessarily, or run other GPU notebooks concurrently.

The worker has a 600-second wall budget; the parent allows 45 additional seconds for teardown. If OOM, timeout, failure or dirty cleanup occurs, stop and inspect the retained partial result. A failed capture is not complete evidence. Verify process/GPU state before retrying. No background server is started.


### Readiness checkpoint

Replace the explanation placeholders with your own short answers before setting RUN_GPU to True. The gate checks completed fields, layer artifacts and frozen choices as well as arithmetic tests. **A filled field is not proof of understanding**: a reviewer should check your explanation against the numbers. No GPU time or package installation is needed to fix an unfinished CPU investigation.


In [ ]:
def require_explanations(answers, name):
    assert isinstance(answers, dict) and answers, name+' must be a nonempty answer map'
    for key, value in answers.items():
        assert isinstance(value, str) and value.strip() and 'TODO' not in value.upper(), name+': finish '+key

def check_gpu_readiness():
    test_scale(derive_scale)
    test_groups(quantize_groups)
    test_summary(summarize_seconds)
    check_lengths(LENGTHS)
    for name, answers in [('PREDICTION', PREDICTION), ('TOY_EXPLANATION', TOY_EXPLANATION),
                          ('METHOD_EXPLANATION', METHOD_EXPLANATION), ('LAYER_EXPLANATION', LAYER_EXPLANATION)]:
        require_explanations(answers, name)
    assert set(CPU_COMPARISON) == {'None','4','2','1'}
    assert len(REAL_RESULTS) == 5 and len(CLIP_RESULTS) == 3
    assert json.loads(FROZEN_CHOICE) == LAYER_DECISION['choice']
    assert json.loads(Path('topic03-layer-decision.json').read_text()) == LAYER_DECISION
    assert CHOSEN_GROUP == LAYER_DECISION['choice']['group'] and CHOSEN_CLIP == LAYER_DECISION['choice']['clipping']
    assert np.isfinite(HELD_OUT_ERROR) and np.isfinite(NF4_TEST_ERROR)
    assert PLOT_GROUPS == [None, CHOSEN_GROUP]
    for filename in ['topic03-layer-decision.json','topic03-weights-full.png',
                     'topic03-weights-center.png','topic03-output-scatter.png']:
        assert Path(filename).is_file() and Path(filename).stat().st_size > 0, 'Missing CPU artifact: '+filename
    require_explanations({key: QUALITY_RULE[key] for key in ['context','rationale']}, 'QUALITY_RULE')
    for key in ['min_correct_of_12','max_invalid_of_12']:
        assert type(QUALITY_RULE[key]) is int and 0 <= QUALITY_RULE[key] <= 12, 'Declare '+key+' before running'
    return json.dumps(QUALITY_RULE, sort_keys=True)


In [ ]:
RUN_GPU = False  # Explicit opt-in only after the CPU tests and prediction are complete.
ENV = Path('/content/lsm-topic03-pilot-env') if Path('/content').exists() else Path('.venv-topic03-pilot')
WORKER_PYTHON = ENV / 'bin' / 'python'
if RUN_GPU:
    ready_rule = check_gpu_readiness()
    if FROZEN_QUALITY_RULE is None:
        FROZEN_QUALITY_RULE = ready_rule
    assert ready_rule == FROZEN_QUALITY_RULE, 'Do not change your pre-run screening rule'
    Path('topic03-preflight.json').write_text(json.dumps(dict(
        quality_rule=json.loads(FROZEN_QUALITY_RULE), layer_choice=json.loads(FROZEN_CHOICE),
        prediction=PREDICTION, toy=TOY_EXPLANATION, methods=METHOD_EXPLANATION,
        layer=LAYER_EXPLANATION, runner_sha256=EXPECTED_RUNNER_SHA256), indent=2))
    print('Pre-install headroom:', infra.resource_sample(require_idle=True, startup=True))
    # Reconfigure this dedicated venv even if a prior ensurepip attempt left a Python binary.
    # No --clear: preserve installed matching packages. Never bootstrap or install into base Python.
    subprocess.check_call([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(ENV)])
    environment_probe = "import json,sys,sysconfig,pip; print(json.dumps({'prefix':sys.prefix,'base_prefix':sys.base_prefix,'purelib':sysconfig.get_paths()['purelib']}))"
    worker_environment = json.loads(subprocess.check_output([str(WORKER_PYTHON), '-c', environment_probe], text=True))
    assert Path(worker_environment['prefix']).resolve() == ENV.resolve(), 'Worker must use the dedicated venv'
    assert worker_environment['prefix'] != worker_environment['base_prefix'], 'Never install into base Python'
    assert Path(worker_environment['purelib']).resolve().is_relative_to(ENV.resolve()), 'Install target must stay inside the venv'
    version_probe = "import importlib.metadata as m,json; names=" + repr(list(infra.PINS)) + "; result={}; "
    version_probe += "\nfor n in names:\n try: result[n]=m.version(n)\n except m.PackageNotFoundError: result[n]=None\nprint(json.dumps(result))"
    installed = json.loads(subprocess.check_output([str(WORKER_PYTHON), '-c', version_probe], text=True))
    missing = [name + '==' + wanted for name, wanted in infra.PINS.items() if installed.get(name) != wanted]
    if missing:
        subprocess.check_call([str(WORKER_PYTHON), '-m', 'pip', '--isolated', 'install', *missing])
    print('Isolated worker packages prepared; host Torch/CUDA will be recorded, not silently replaced.')
else:
    print('GPU and package installation are OFF. Complete the CPU work first.')


In [ ]:
from datetime import datetime, timezone
ATTEMPT = None
if RUN_GPU:
    check_lengths(LENGTHS)
    assert check_gpu_readiness() == FROZEN_QUALITY_RULE
    ATTEMPT = Path('evidence') / datetime.now(timezone.utc).strftime('topic03-pilot-%Y%m%dT%H%M%S%fZ')
    ATTEMPT.mkdir(parents=True, exist_ok=False)
    command = [str(WORKER_PYTHON), str(RUNNER), '--output', str(ATTEMPT/'result.json'),
               '--lengths', *map(str, LENGTHS), '--trials', str(TRIALS), '--timeout', '600']
    subprocess.run(command, check=True, timeout=645)
    # Separate post-process query; do not publish account/PID details from the notebook.
    print(subprocess.check_output(['nvidia-smi', '--id=0', '--query-gpu=memory.used',
          '--format=csv,noheader,nounits'], text=True).strip(), 'MiB device memory after worker exit')
    infra.resource_sample(require_idle=True)
    print('Preserved attempt:', ATTEMPT)
else:
    print('Not run: no new GPU evidence exists.')


## C1. Compare the provided summary with the retained raw evidence

Now apply the provided timing function to every format/length group. Preserve seconds in the raw artifact and label the converted summary in milliseconds. Inspect natural quality answers separately.


In [ ]:
EVIDENCE_FILE = ATTEMPT/'result.json' if ATTEMPT is not None else None
RESULT = None
if EVIDENCE_FILE is not None:
    RESULT = json.loads(EVIDENCE_FILE.read_text())
    assert infra.validate_artifact(RESULT)
    for label, entry in RESULT['formats'].items():
        for length in LENGTHS:
            rows = [row for row in entry['timing_trials'] if row['length'] == length]
            own = summarize_seconds([row['wall_s'] for row in rows])
            official = entry['summaries'][str(length)]
            assert all(math.isclose(value, official[key], rel_tol=1e-12) for key, value in own.items())
            print(label, length, 'raw seconds:', [row['wall_s'] for row in rows])
            print('Verified summary:', own)
            print('Peak allocated MiB:', official['peak_allocated_bytes']/2**20,
                  'peak reserved MiB:', official['peak_reserved_bytes']/2**20)
        print('Natural quality:', entry['quality_summary'])
        columns = list(infra.LABELS) + ['invalid']
        confusion = [[sum(r['expected_label']==truth and
            (r['predicted_label'] if r['valid_label'] else 'invalid')==pred
            for r in entry['quality_samples']) for pred in columns] for truth in infra.LABELS]
        print('Confusion rows=true, columns=predicted', columns, confusion)
        fig, ax = plt.subplots(figsize=(5,3))
        ax.imshow(confusion, cmap='Blues', vmin=0, vmax=4)
        ax.set_xticks(range(4), columns); ax.set_yticks(range(3), infra.LABELS)
        ax.set(xlabel='Predicted label', ylabel='True label', title=label+' synthetic smoke (n=12)')
        for i,row in enumerate(confusion):
            for j,count in enumerate(row): ax.text(j,i,str(count),ha='center',va='center')
        fig.tight_layout(); fig.savefig('topic03-confusion-'+label+'.png', dpi=150); plt.show()
        for row in entry['quality_samples']:
            print(row['case_id'], repr(row['output']), row['stop_reason'],
                  'valid:', row['valid_label'], 'correct:', row['correct'])
    assert json.dumps(QUALITY_RULE, sort_keys=True) == FROZEN_QUALITY_RULE
    screening = {}
    for label, entry in RESULT['formats'].items():
        quality = entry['quality_summary']
        correct = quality['correct_count']
        invalid = sum(not r['valid_label'] for r in entry['quality_samples'])
        screening[label] = dict(correct=correct, invalid=invalid, passed=(
            correct >= QUALITY_RULE['min_correct_of_12'] and invalid <= QUALITY_RULE['max_invalid_of_12']))
    Path('topic03-screening.json').write_text(json.dumps(dict(rule=json.loads(FROZEN_QUALITY_RULE),
        results=screening, scope='classroom synthetic smoke; not deployment acceptance'), indent=2))
    print('Predeclared screening result:', screening)
    fig, ax = plt.subplots(figsize=(6,4))
    for label, entry in RESULT['formats'].items():
        for length in LENGTHS:
            values = [row['wall_s']*1000 for row in entry['timing_trials'] if row['length']==length]
            ax.scatter([label+'/'+str(length)]*len(values), values, s=18)
    ax.set(xlabel='Format / forced output IDs', ylabel='Synchronized generate wall time (ms)',
           title='Five raw trials per condition; no p95 claim')
    fig.tight_layout(); fig.savefig('topic03-timing.png', dpi=150); plt.show()
    for fp,nf in zip(RESULT['formats']['fp16']['quality_samples'], RESULT['formats']['nf4']['quality_samples']):
        assert fp['case_id']==nf['case_id']
        if fp['predicted_label'] != nf['predicted_label']:
            print('Paired change:', fp['case_id'], 'true', fp['expected_label'],
                  fp['predicted_label'], '->', nf['predicted_label'])
else:
    print('No artifact loaded. Do not replace your run with fabricated numbers.')


## C2. Read a profile without inventing a cause

The runner takes one 24-ID profile per format **after** its timing trials. Examine `profile.status` first. If unavailable, report that limitation. If complete, find an operator that changes and write one testable hypothesis. Inclusive CPU/device times can overlap; summing them does not produce wall time. A `Linear4bit` module count verifies a treatment, not an accelerated kernel.

The profiler summary is not a measurement of HBM bandwidth, achieved occupancy or pure dequantization overhead. It does not, by itself, prove why the wall time changed. Reserved allocator pools persist after warmup and between lengths: they are not independent fresh-memory experiments.


In [ ]:
if RESULT is not None:
    for label, entry in RESULT['formats'].items():
        print(label, 'quantized linear modules:', entry['quantized_linear_modules'])
        print(json.dumps(entry['profile'], indent=2))
else:
    print('No profile to interpret; do not infer kernels from the toy or module names.')


## C3. Submit a bounded decision, not a victory slide

Keep your modified notebook (or a readable diff), `topic03-layer-decision.json`, matched histograms and output scatter, original GPU attempt `result.json`, one timing plot/table and a half-page decision memo. Explain the frozen calibration choice and its held-out error separately from GPU time, memory and ticket accuracy. This pilot is a teaching artifact, not a replacement for existing judge submission files.

**Before disconnecting Colab:** download all items in the checklist below through the Files pane (right-click → Download); use File → Download → Download .ipynb for your edited notebook. Colab's temporary VM is not durable storage. Keep runner/model/runtime metadata with the JSON. Check outputs for tokens, private paths or account data before sharing; submit privately, not as a public Issue. Do not download the model cache. Preserve failed attempts separately, clearly marked partial rather than completed evidence.

Your memo must state:

1. One prediction checked against evidence: supported, revised or unresolved. A correct prediction need not become wrong.
2. The decision: keep FP16 as control, keep NF4 conditionally, reject both for the application, or postpone. Cite the frozen screening rule, a time/memory result and the layer's held-out error, keeping their scopes separate.
3. One limitation and one discriminating next experiment. A correctly rejected optimization is a successful investigation.

Keep the memo about half a page. **Detailed grouping tables, AWQ/GPTQ answers, all ticket mismatches, runtime identity and raw trials belong in notebook answers and attachments, not repeated in the memo.** Use the [short decision template](https://levshaazz.github.io/llm-serving-mastery/downloads/topic03-decision-template.md).

### Save checklist

- Edited notebook, including predictions and all explanation fields.
- `topic03-layer-decision.json`, `topic03-weights-full.png`, `topic03-weights-center.png`, `topic03-output-scatter.png`.
- `topic03-preflight.json` (the pre-run screening rule and answers), `topic03-screening.json` (its result).
- Original `evidence/<your attempt>/result.json`, `topic03-timing.png`, `topic03-confusion-fp16.png`, `topic03-confusion-nf4.png`.
- Your `topic03-decision.md` memo. Save any failed attempt's partial JSON separately. Missing GPU access must be reported as **not run**, not replaced by teacher numbers.

Check that downloaded files actually open before ending the runtime.


**Transfer check:** before touching the six public transfer tickets in `infra.TRANSFER_CASES`, write what failure you expect; do not tune the core prompt on them. They are an exercise, not a hidden grading set. An optional extension may run them separately after the required artifact is preserved. No automatic extra GPU run is required.

### Optional, not hidden mandatory work

Full AWQ/GPTQ implementations and derivations, detailed NF4 codebook construction, double quantization, FP8/NVFP4, full packing implementation, reverse-format-order replication and custom kernels are outside this pilot's mandatory core. The 15-topic course framework remains; overload is not moved into mandatory homework.


## Sources and status

- Pedagogical idea: [MIT EfficientML Lab 4](https://colab.research.google.com/drive/16H9RvSg4XIF35X3fLGQUVwAE9ccvDj14), controlled comparisons. Its pseudo-quantization example does not demonstrate packed storage or kernel acceleration. Our code and support cases are original.
- Pedagogical idea: [CMU Deep Learning Systems](https://dlsyscourse.org/), small student functions inside a cumulative project. We do not adopt its full prerequisite level.
- Runtime behavior: [Transformers 4.51.3 bitsandbytes guide](https://huggingface.co/docs/transformers/v4.51.3/en/quantization/bitsandbytes), [bitsandbytes 0.50.2 source](https://github.com/bitsandbytes-foundation/bitsandbytes/tree/0.50.2), [PyTorch CUDA memory semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html#cuda-memory-management), [PyTorch profiler](https://docs.pytorch.org/docs/stable/profiler.html).
- The [2 October historical v3 capture](https://levshaazz.github.io/llm-serving-mastery/seminars/runs/2026-10-02-topic03-quantization/README.md) used a different four-prompt protocol and instructor hardware. It is not this pilot, not a comparison against your Colab device, and not your execution.

This is a pilot awaiting learner evaluation. Automated checks can verify arithmetic, identity and artifact completeness; they cannot prove that a student can independently transfer the skill.
